# DIF Detection in Rasch Measurement Model (RMM)
## Differential Item Functioning: 1반 vs 2반

### Table of Contents
1. [What is DIF? — Causes in Two Classrooms](#sec1)
2. [DIF Detection Methods Overview](#sec2)
3. [Simulation: Two Groups with Planted DIF](#sec3)
4. [Method 1 — Separate Rasch Calibration (b-Difference)](#sec4)
5. [Method 2 — Mantel-Haenszel Test](#sec5)
6. [Method 3 — Likelihood Ratio Test (LRT)](#sec6)
7. [Method 4 — Bayesian DIF (Stan)](#sec7)
8. [Summary and DIF Classification](#sec8)


In [ ]:
import sys as _sys, os as _os
import matplotlib as _mpl, matplotlib.font_manager as _fm

def _setup_korean_font():
    _c = {
        'win32':  [('C:/Windows/Fonts/malgun.ttf','Malgun Gothic'),
                   ('C:/Windows/Fonts/gulim.ttc','Gulim')],
        'darwin': [('/System/Library/Fonts/AppleSDGothicNeo.ttc','Apple SD Gothic Neo')],
        'linux':  [('/usr/share/fonts-droid-fallback/truetype/DroidSansFallback.ttf',
                    'Droid Sans Fallback')],
    }
    _fm.fontManager.ttflist = [f for f in _fm.fontManager.ttflist
        if not (f.name=='Droid Sans Fallback' and 'Full' in f.fname)]
    for path, name in _c.get(_sys.platform, _c['linux']):
        if _os.path.exists(path):
            _fm.fontManager.addfont(path)
            _mpl.rcParams['font.family'] = ['DejaVu Sans', name]
            return
    _mpl.rcParams['font.family'] = 'DejaVu Sans'

_setup_korean_font()
_mpl.rcParams['axes.unicode_minus'] = False

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import minimize, minimize_scalar
from scipy.stats import chi2, norm
import warnings, os, tempfile
warnings.filterwarnings('ignore')
np.random.seed(42)
tmpdir = tempfile.mkdtemp()

try:
    import cmdstanpy; STAN_AVAILABLE = True; print("Stan available.")
except ImportError:
    cmdstanpy = None; STAN_AVAILABLE = False
    print("cmdstanpy not available — Bayesian DIF uses simulated posterior.")


---
<a id="sec1"></a>
## 1. What is DIF? — All Possible Causes When Comparing Two Classes

### Definition

**Differential Item Functioning (DIF)** occurs when an item behaves **differently for two groups after controlling for overall ability** (latent trait θ):

$$\text{DIF on item } i: \quad P(X_{ji}=1 \mid \theta, G=1) \neq P(X_{ji}=1 \mid \theta, G=2)$$

In the Rasch model this means the item difficulty differs between groups:

$$b_i^{(G1)} \neq b_i^{(G2)}$$

A DIF item is **biased** if the source of the difference is construct-irrelevant (e.g., unfair wording).  
A DIF item is **impact** if the difference reflects a real, substantive group difference on a secondary trait.

---

### All Possible Reasons DIF Can Occur Between Class 1 and Class 2

#### A. Teaching and Curriculum Differences (교수·교육과정 차이)

| # | Cause | Mechanism |
|---|---|---|
| A1 | **Different instructional emphasis** | Teacher 1 drilled item-type X; Teacher 2 did not → item X is easier for Class 1 |
| A2 | **Differential curriculum coverage** | A topic was taught in different depth, order, or style |
| A3 | **Different textbooks or materials** | Problem formats in one book match test items more closely |
| A4 | **Review session / coaching** | One class received a pre-test review covering some item types |
| A5 | **Worked example exposure** | Certain items resemble homework problems seen only by one class |
| A6 | **Conceptual vs. procedural emphasis** | Items requiring computation favour a procedurally taught class; conceptual items favour the other |

#### B. Test Administration Differences (시험 시행 차이)

| # | Cause | Mechanism |
|---|---|---|
| B1 | **Testing time / time of day** | Morning vs. afternoon testing introduces fatigue differences |
| B2 | **Different test administrators / proctors** | One proctor read instructions differently or answered student questions |
| B3 | **Different testing room conditions** | Noise, lighting, seating density affect concentration |
| B4 | **Item order effects** | If tests were administered in different orders, carry-over and fatigue differ |
| B5 | **Time pressure** | If one class had less time, speeded conditions create DIF on harder items |
| B6 | **Presence/absence of scratch paper, calculators** | Affects items requiring computation |

#### C. Item Content and Format Effects (문항 내용·형식 효과)

| # | Cause | Mechanism |
|---|---|---|
| C1 | **Context-specific items** | Items set in contexts (sports, cooking, technology) familiar to one class |
| C2 | **Linguistic complexity** | Items with complex sentence structures disadvantage lower-reading-ability groups |
| C3 | **Cultural references** | Items assuming specific cultural knowledge (holidays, customs, media) |
| C4 | **Gender-related content** | Items about stereotypically gendered topics if classes differ in gender composition |
| C5 | **Ambiguous wording** | An ambiguous item can be interpreted differently by students exposed to different teaching |
| C6 | **Real-world application context** | If the context was explicitly taught in one class (e.g., applied maths problem) |

#### D. Student Group Characteristic Differences (학생 집단 특성 차이)

| # | Cause | Mechanism |
|---|---|---|
| D1 | **Ability distribution differences** | If one class has systematically higher ability, anchoring/linking issues can mimic DIF |
| D2 | **Motivation and test-taking effort** | One class is less motivated (e.g., non-graded test for one group) |
| D3 | **Test-taking strategy training** | One class was taught to eliminate wrong answers, use time management, guess strategically |
| D4 | **Prior knowledge from other courses** | Cross-disciplinary knowledge relevant to items |
| D5 | **Language proficiency differences** | If classes differ in language background (bilingual, EFL, etc.) |
| D6 | **Special needs / accommodations** | One class has a higher proportion of students with learning differences |

#### E. Measurement Model Violations (측정 모형 위반)

| # | Cause | Mechanism |
|---|---|---|
| E1 | **Multidimensionality** | Test measures 2 traits; one class is stronger on the secondary trait |
| E2 | **Local item dependence** | Item cluster from one topic; one class studied that topic exclusively |
| E3 | **Item discrimination differences** | If one group has more variable ability, discrimination effectively differs (violates Rasch) |
| E4 | **Differential guessing** | Higher-ability group guesses better on ambiguous items |
| E5 | **Ceiling / floor effects per group** | If one group clusters at extreme scores, Rasch estimates are unstable |
| E6 | **Model misspecification** | True model is 2PL but Rasch is fitted — DIF appears because discrimination is ignored |

#### F. Sampling and Statistical Artefacts (표집·통계적 인공산물)

| # | Cause | Mechanism |
|---|---|---|
| F1 | **Small class size** | Random sampling variation in N≈30 inflates apparent DIF |
| F2 | **Non-equivalent anchor items** | DIF detection relies on non-DIF anchor items; if anchors are themselves DIF, all estimates shift |
| F3 | **Ability distribution mismatch** | Groups cover different θ ranges → interpolation errors in Mantel-Haenszel strata |
| F4 | **Purification failure** | DIF items included in the anchor contaminate the linking scale |

---

### (한국어) 1반과 2반 간 DIF 발생 가능한 원인 요약

DIF는 능력(θ)을 통제한 후에도 두 집단에서 문항이 **다르게 기능**할 때 발생합니다.  
같은 학교의 두 학급을 비교할 때 DIF의 주요 원인은 다음과 같습니다:

**A. 교수·교육과정 차이**: 담당 교사가 다를 경우 특정 유형의 문항에 대한 학습 경험이 달라집니다.  
특정 선생님이 유사한 문제를 많이 다뤘다면 해당 문항은 그 학급에 유리합니다.

**B. 시험 시행 조건**: 오전/오후 시험, 다른 감독관, 시험실 환경, 시간 압박 등  
시험 시행 방식의 차이가 특정 문항에 대한 반응에 영향을 미칩니다.

**C. 문항 내용**: 특정 맥락(스포츠, 문화, 성별)을 전제하는 문항은 해당 경험이 있는 집단에 유리합니다.

**D. 학생 집단 특성**: 학습 동기, 시험 전략, 사전 지식, 언어 능력 차이.

**E. 측정 모형 위반**: 검사가 두 가지 특성을 측정하거나(다차원성), 문항 간 종속성이 있거나,  
실제 변별도가 집단 간에 다른 경우.

**F. 통계적 인공산물**: 소표본(N≈30)에서의 무선 변동, 앵커 문항 오염, 능력 범위 불일치.

> **핵심 원칙**: DIF 탐지는 두 반의 *점수 차이(impact)*를 측정하는 것이 아닙니다.  
> θ를 통제한 후에도 남는 *문항별 집단 차이*를 탐지합니다.


---
<a id="sec2"></a>
## 2. DIF Detection Methods Overview

### Methods Implemented in This Notebook

| Method | Type | What it tests | Assumptions |
|---|---|---|---|
| **b-Difference (Separate Calibration)** | Rasch-based | $\Delta b_i = \hat b_i^{G1} - \hat b_i^{G2}$ after linking | Rasch fit in each group |
| **Mantel-Haenszel (MH)** | Non-parametric | OR of correct response across matched-score strata | Common OR across strata |
| **Likelihood Ratio Test (LRT)** | Rasch-based | $-2 \log \Lambda$ comparing constrained vs. free item models | Rasch fit overall |
| **Bayesian Posterior Comparison** | Bayesian | Posterior of $b_i^{G1} - b_i^{G2}$; credible interval excludes 0 | Normal priors on b |

### DIF Magnitude Classification (ETS Scheme)

| Category | $|\Delta b|$ (logits) | $\text{MH } |\Delta|$ | Interpretation |
|---|---|---|---|
| **A** (negligible) | < 0.43 | < 1.0 | No meaningful DIF |
| **B** (moderate) | 0.43–0.64 | 1.0–1.5 | Review item; use with caution |
| **C** (large) | > 0.64 | > 1.5 | Item should be removed or revised |

### Rasch b-Difference Method

After separate calibration and **mean-mean linking**:

$$\Delta b_i = \hat b_i^{(G1)} - (\hat b_i^{(G2)} + c)$$

where $c = \bar{b}^{(G1)} - \bar{b}^{(G2)}$ is the linking constant (mean-centering alignment).

An item shows DIF if $|\Delta b_i|$ exceeds the critical value:

$$|\Delta b_i| > 1.96 \sqrt{\text{SE}_{b_i^{G1}}^2 + \text{SE}_{b_i^{G2}}^2}$$

### Mantel-Haenszel Statistic

At each score stratum $s$, form the 2×2 table:

|  | Correct | Incorrect |
|---|---|---|
| Group 1 | $A_s$ | $B_s$ |
| Group 2 | $C_s$ | $D_s$ |

$$\chi^2_{MH} = \frac{\left(\left|\sum_s A_s - \sum_s \frac{(A_s+B_s)(A_s+C_s)}{N_s}\right| - 0.5\right)^2}{\sum_s \frac{(A_s+B_s)(C_s+D_s)(A_s+C_s)(B_s+D_s)}{N_s^2(N_s-1)}}$$

Under $H_0$ (no DIF), $\chi^2_{MH} \sim \chi^2(1)$.

The MH log-odds ratio:

$$\hat\alpha_{MH} = \frac{\sum_s A_s D_s / N_s}{\sum_s B_s C_s / N_s}$$

$\text{MH} |\Delta| = -2.35 \ln(\hat\alpha_{MH})$: positive = item harder for Group 1; negative = harder for Group 2.

### Likelihood Ratio Test

Fit two Rasch models:
- **Constrained**: same item difficulties across groups
- **Free**: item $i$'s difficulty is a free parameter in each group

$$\Lambda = -2(\ell_{constrained} - \ell_{free}) \sim \chi^2(I)$$

For item-level LRT (test one item at a time for DIF):

$$\Lambda_i = -2(\ell_{constrained} - \ell_{free,i}) \sim \chi^2(1)$$


In [ ]:
# ─── Section 3: Simulation — Two Groups with Planted DIF ──────────────────
np.random.seed(42)

J1, J2 = 40, 40   # class sizes
I = 20             # items

# ── True item difficulties (shared base) ──
b_base = np.linspace(-2.0, 2.0, I)
b_base -= b_base.mean()

# ── Plant DIF on 4 items ──
# Positive Δb: item is harder for Group 1 (favours Group 2)
# Negative Δb: item is easier for Group 1 (favours Group 1)
DIF_items  = np.array([2, 5, 11, 16])      # 0-indexed
DIF_deltas = np.array([+0.8, -0.9, +1.2, -0.7])  # logit shift FOR GROUP 1

b_true_G1 = b_base.copy()
b_true_G2 = b_base.copy()
for idx, delta in zip(DIF_items, DIF_deltas):
    b_true_G1[idx] += delta / 2
    b_true_G2[idx] -= delta / 2

# ── Person abilities ──
# Class 1 slightly lower mean (realistic: heterogeneous classes)
theta_G1 = np.random.normal(-0.2, 1.0, J1)
theta_G2 = np.random.normal( 0.2, 1.0, J2)

# ── Generate responses ──
def gen_responses(theta, b):
    J, I = len(theta), len(b)
    logit = theta[:, None] - b[None, :]
    P = 1.0 / (1.0 + np.exp(-logit))
    return (np.random.uniform(size=(J, I)) < P).astype(int)

X1 = gen_responses(theta_G1, b_true_G1)
X2 = gen_responses(theta_G2, b_true_G2)

print("=== Simulated Data ===")
print(f"Class 1: {J1} persons  |  scores: {X1.sum(axis=1).min()}–{X1.sum(axis=1).max()}"
      f"  mean={X1.sum(axis=1).mean():.1f}")
print(f"Class 2: {J2} persons  |  scores: {X2.sum(axis=1).min()}–{X2.sum(axis=1).max()}"
      f"  mean={X2.sum(axis=1).mean():.1f}")
print(f"\nDIF items (planted): {DIF_items + 1}  (1-indexed)")
print(f"DIF deltas (G1-G2) : {DIF_deltas}")

# Quick overview plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Simulated Data — Class 1 vs Class 2', fontsize=13, fontweight='bold')

axes[0].hist(theta_G1, bins=15, color='steelblue', alpha=0.6, label='Class 1', density=True)
axes[0].hist(theta_G2, bins=15, color='firebrick', alpha=0.6, label='Class 2', density=True)
axes[0].set_title('Person Ability Distribution (true θ)', fontsize=11)
axes[0].set_xlabel('Logit'); axes[0].legend()

axes[1].bar(np.arange(I)+0.8, b_true_G1, width=0.4, color='steelblue', alpha=0.7, label='Class 1 b')
axes[1].bar(np.arange(I)+1.2, b_true_G2, width=0.4, color='firebrick', alpha=0.7, label='Class 2 b')
for d in DIF_items:
    axes[1].axvline(d+1, color='gold', linewidth=2, alpha=0.8, zorder=0)
axes[1].set_title('True Item Difficulties (DIF items = yellow)', fontsize=11)
axes[1].set_xlabel('Item'); axes[1].set_ylabel('b (logit)'); axes[1].legend(fontsize=8)

axes[2].hist(X1.sum(axis=1), bins=range(0, I+2), color='steelblue', alpha=0.6,
             label='Class 1', density=True)
axes[2].hist(X2.sum(axis=1), bins=range(0, I+2), color='firebrick', alpha=0.6,
             label='Class 2', density=True)
axes[2].set_title('Total Score Distribution', fontsize=11)
axes[2].set_xlabel('Score'); axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'dif_data.png'), dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ─── Shared CML Functions ───────────────────────────────────────────────────
def compute_esp(eps, I):
    # Elementary symmetric polynomials via sum algorithm
    gamma = np.zeros(I + 1); gamma[0] = 1.0
    for k in range(I):
        for r in range(min(k+1, I), 0, -1):
            gamma[r] += gamma[r-1] * eps[k]
    return gamma

def cml_nll(b, X):
    J, I = X.shape
    eps = np.exp(-b); gamma = compute_esp(eps, I)
    rscores = X.sum(axis=1).astype(int)
    s = X.sum(axis=0)
    nll = np.dot(b, s)
    for j in range(J):
        r = rscores[j]
        if 0 < r < I:
            nll -= np.log(gamma[r] + 1e-300)
    return nll

def fit_rasch_cml(X):
    I = X.shape[1]
    res = minimize(cml_nll, np.zeros(I), args=(X,), method='L-BFGS-B',
                   options={'maxiter':2000,'ftol':1e-12})
    b = res.x - res.x.mean()
    # SE from diagonal Hessian
    eps_h = 1e-4
    hd = np.array([(cml_nll(b + eps_h*np.eye(I)[i], X)
                    - 2*cml_nll(b, X)
                    + cml_nll(b - eps_h*np.eye(I)[i], X)) / eps_h**2
                   for i in range(I)])
    se = 1.0 / np.sqrt(np.abs(hd) + 1e-10)
    return b, se

def theta_mle(b, x_j):
    r = x_j.sum()
    if r == 0: return -5.0
    if r == len(b): return 5.0
    def nll(th):
        p = 1/(1+np.exp(-(th-b)))
        return -(x_j@np.log(p+1e-300)+(1-x_j)@np.log(1-p+1e-300))
    return minimize_scalar(nll, bounds=(-6,6), method='bounded').x

print("CML functions loaded. Fitting models...")
b_G1, se_G1 = fit_rasch_cml(X1)
b_G2, se_G2 = fit_rasch_cml(X2)
print(f"Group 1 b: mean={b_G1.mean():.3f}  SD={b_G1.std():.3f}")
print(f"Group 2 b: mean={b_G2.mean():.3f}  SD={b_G2.std():.3f}")


In [ ]:
# ─── Section 4: Rasch b-Difference DIF Method ──────────────────────────────

# ── Mean-mean linking: adjust G2 scale to G1 ──
linking_const = b_G1.mean() - b_G2.mean()
b_G2_linked   = b_G2 + linking_const

# ── DIF statistic: delta_b ──
delta_b   = b_G1 - b_G2_linked
delta_se  = np.sqrt(se_G1**2 + se_G2**2)        # pooled SE
z_scores  = delta_b / delta_se
p_values  = 2 * (1 - norm.cdf(np.abs(z_scores))) # two-tailed

# ── ETS magnitude classification ──
def ets_class(d):
    ad = abs(d)
    if ad < 0.43:  return 'A'
    elif ad < 0.64: return 'B'
    else:           return 'C'

ets_labels = [ets_class(d) for d in delta_b]
colors_dif  = ['#d62728' if c=='C' else '#ff7f0e' if c=='B' else 'steelblue'
               for c in ets_labels]

print(f"{'Item':>5} {'b_G1':>7} {'b_G2_lnk':>9} {'Δb':>7} {'SE':>6} "
      f"{'z':>6} {'p':>7} {'ETS':>4} {'True DIF':>9}")
print("-" * 72)
for i in range(I):
    true_dif = DIF_deltas[DIF_items.tolist().index(i)] if i in DIF_items else 0.0
    flag = " ◀ DIF" if i in DIF_items else ""
    print(f"{i+1:5d} {b_G1[i]:7.3f} {b_G2_linked[i]:9.3f} "
          f"{delta_b[i]:7.3f} {delta_se[i]:6.3f} "
          f"{z_scores[i]:6.2f} {p_values[i]:7.4f} "
          f"{ets_labels[i]:>4}  {true_dif:+.2f}{flag}")

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Rasch b-Difference DIF Detection (1반 vs 2반)', fontsize=13, fontweight='bold')

# Panel A: b_G1 vs b_G2 scatter
ax = axes[0]
ax.scatter(b_G2_linked, b_G1, c=colors_dif, s=80, zorder=3, edgecolors='white')
lo = min(b_G2_linked.min(), b_G1.min()) - 0.3
hi = max(b_G2_linked.max(), b_G1.max()) + 0.3
ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, alpha=0.4)
for i in range(I):
    ax.annotate(f'I{i+1}', (b_G2_linked[i], b_G1[i]),
                fontsize=7, color='gray', xytext=(3,3), textcoords='offset points')
ax.set_xlabel('b̂ Group 2 (linked)', fontsize=11); ax.set_ylabel('b̂ Group 1', fontsize=11)
ax.set_title('Item Difficulty: Group 1 vs Group 2\n(red=C-DIF, orange=B-DIF, blue=A)', fontsize=10)

# Panel B: Δb with CI
ax2 = axes[1]
items = np.arange(1, I+1)
ax2.bar(items, delta_b, color=colors_dif, alpha=0.75, edgecolor='white')
ax2.errorbar(items, delta_b, yerr=1.96*delta_se,
             fmt='none', color='black', capsize=4, linewidth=1.2)
ax2.axhline(0,   color='black', linewidth=1.2, linestyle='--')
ax2.axhline( 0.64, color='red',   linewidth=1.0, linestyle=':', alpha=0.7, label='C threshold ±0.64')
ax2.axhline(-0.64, color='red',   linewidth=1.0, linestyle=':', alpha=0.7)
ax2.axhline( 0.43, color='orange',linewidth=1.0, linestyle=':', alpha=0.7, label='B threshold ±0.43')
ax2.axhline(-0.43, color='orange',linewidth=1.0, linestyle=':', alpha=0.7)
ax2.set_xlabel('Item', fontsize=11); ax2.set_ylabel('Δb = b_G1 − b_G2 (logit)', fontsize=11)
ax2.set_title('b-Difference DIF Statistic ± 1.96 SE', fontsize=11)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'dif_bdiff.png'), dpi=120, bbox_inches='tight')
plt.show()

detected = [i+1 for i in range(I) if ets_labels[i] != 'A']
print(f"\nb-Difference DIF detected (B or C): items {detected}")
true_items = [d+1 for d in DIF_items]
print(f"True DIF items: {true_items}")


In [ ]:
# ─── Section 5: Mantel-Haenszel DIF Test ───────────────────────────────────

def mantel_haenszel(X1, X2, item_idx):
    # Stratify by total score (excluding focal item for purer matching)
    scores1 = X1.sum(axis=1)  # total scores Group 1
    scores2 = X2.sum(axis=1)  # total scores Group 2

    all_scores = np.concatenate([scores1, scores2])
    min_s, max_s = int(all_scores.min()), int(all_scores.max())

    num = 0.0; denom = 0.0
    A_obs_total = 0.0; A_exp_total = 0.0; var_total = 0.0
    alpha_num = 0.0; alpha_den = 0.0

    for s in range(min_s, max_s + 1):
        g1_mask = (scores1 == s)
        g2_mask = (scores2 == s)
        n1s = g1_mask.sum(); n2s = g2_mask.sum()
        if n1s == 0 or n2s == 0:
            continue
        Ns = n1s + n2s
        A  = X1[g1_mask, item_idx].sum()   # G1 correct
        B  = n1s - A                        # G1 incorrect
        C  = X2[g2_mask, item_idx].sum()   # G2 correct
        D  = n2s - C                        # G2 incorrect

        A_exp = n1s * (A + C) / Ns
        var_s = (n1s * n2s * (A+C) * (B+D)) / (Ns**2 * (Ns - 1)) if Ns > 1 else 0

        A_obs_total += A
        A_exp_total += A_exp
        var_total   += var_s
        alpha_num   += A * D / Ns
        alpha_den   += B * C / Ns

    if var_total < 1e-10 or alpha_den < 1e-10:
        return np.nan, np.nan, np.nan, np.nan

    # MH chi-square (with continuity correction)
    chi2_mh = (abs(A_obs_total - A_exp_total) - 0.5)**2 / var_total
    p_mh    = 1 - chi2.cdf(chi2_mh, df=1)

    # MH log-odds ratio and delta
    alpha_mh    = alpha_num / alpha_den
    mh_delta    = -2.35 * np.log(alpha_mh)   # ETS delta scale

    return chi2_mh, p_mh, alpha_mh, mh_delta

print(f"{'Item':>5} {'chi2_MH':>9} {'p_MH':>8} {'OR_MH':>7} "
      f"{'MH_Δ':>7} {'ETS':>4} {'True DIF':>9}")
print("-" * 64)

mh_chi2  = np.zeros(I); mh_p = np.zeros(I)
mh_or    = np.zeros(I); mh_delta = np.zeros(I)

for i in range(I):
    c2, p, orr, mhd = mantel_haenszel(X1, X2, i)
    mh_chi2[i] = c2; mh_p[i] = p; mh_or[i] = orr; mh_delta[i] = mhd
    ets = 'A' if abs(mhd) < 1.0 else ('B' if abs(mhd) < 1.5 else 'C')
    true_dif = DIF_deltas[DIF_items.tolist().index(i)] if i in DIF_items else 0.0
    flag = " ◀" if i in DIF_items else ""
    print(f"{i+1:5d} {c2:9.3f} {p:8.4f} {orr:7.3f} {mhd:7.3f}  {ets:>3}  {true_dif:+.2f}{flag}")

# Plot MH delta
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Mantel-Haenszel DIF Test (1반 vs 2반)', fontsize=13, fontweight='bold')

mh_colors = ['#d62728' if abs(d)>=1.5 else '#ff7f0e' if abs(d)>=1.0 else 'steelblue'
             for d in mh_delta]

ax = axes[0]
ax.bar(np.arange(1, I+1), mh_delta, color=mh_colors, alpha=0.75, edgecolor='white')
ax.axhline(0,    color='black', linewidth=1.2, linestyle='--')
ax.axhline( 1.5, color='red',    linewidth=1.0, linestyle=':', alpha=0.7, label='C: |Δ|=1.5')
ax.axhline(-1.5, color='red',    linewidth=1.0, linestyle=':', alpha=0.7)
ax.axhline( 1.0, color='orange', linewidth=1.0, linestyle=':', alpha=0.7, label='B: |Δ|=1.0')
ax.axhline(-1.0, color='orange', linewidth=1.0, linestyle=':', alpha=0.7)
ax.set_xlabel('Item'); ax.set_ylabel('MH Δ  (positive = harder for G1)'); ax.legend(fontsize=9)
ax.set_title('MH Delta Scale DIF (|Δ|≥1.5=C, ≥1.0=B)', fontsize=11)

ax2 = axes[1]
neg_log_p = -np.log10(np.maximum(mh_p, 1e-10))
ax2.bar(np.arange(1, I+1), neg_log_p, color=mh_colors, alpha=0.75, edgecolor='white')
ax2.axhline(-np.log10(0.05), color='orange', linestyle='--', linewidth=1.2,
            label='p = 0.05')
ax2.axhline(-np.log10(0.001), color='red', linestyle='--', linewidth=1.2,
            label='p = 0.001')
ax2.set_xlabel('Item'); ax2.set_ylabel('−log₁₀(p)'); ax2.legend(fontsize=9)
ax2.set_title('MH Significance (−log₁₀ p)', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'dif_mh.png'), dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ─── Section 6: Likelihood Ratio Test (LRT) for DIF ────────────────────────

# Stack both groups for joint analysis
X_all   = np.vstack([X1, X2])
group   = np.array([0]*len(X1) + [1]*len(X2))   # 0=G1, 1=G2

def cml_nll_joint(b, X):
    # Standard CML NLL (used for constrained model — same b for both groups)
    return cml_nll(b, X)

def cml_nll_free_item(b_full, X, X_g1, X_g2, item_idx):
    # Free model for item item_idx: b_g1[item_idx] != b_g2[item_idx]
    # b_full layout: b_common (I params) + delta_i (1 extra param for the focal item)
    I = X_g1.shape[1]
    b_base  = b_full[:I]
    delta_i = b_full[I]                  # extra shift for G2 on focal item
    b_g2_adj = b_base.copy(); b_g2_adj[item_idx] += delta_i
    return cml_nll(b_base, X_g1) + cml_nll(b_g2_adj, X_g2)

print("LRT item-level DIF (takes ~1 min for 20 items)...")

# Null model: fit on combined data (constrained — same b for both groups)
b0   = np.zeros(I)
res0 = minimize(lambda b: cml_nll(b, X_g1=None) or cml_nll_joint(b, X1) + cml_nll_joint(b, X2),
                b0, method='L-BFGS-B', options={'maxiter':2000,'ftol':1e-12})

# Actually fit constrained model = separate CML for each group but fixed difference
b_null = np.zeros(I)
nll_null_g1 = cml_nll(b_G1, X1)
nll_null_g2 = cml_nll(b_G2, X2)
nll_null_total = nll_null_g1 + nll_null_g2   # baseline (fully free by group, no DIF constraint)

lrt_chi2 = np.zeros(I); lrt_p = np.zeros(I)

for focal in range(I):
    # Constrained for this item: b_G1[focal] == b_G2[focal] (force equality on focal item)
    # Test: does freeing b[focal] per group significantly improve fit?

    # Free model NLL (already computed: separate CML for each group)
    nll_free = nll_null_total   # fully free per group

    # Constrained NLL: force b_G1[focal] = b_G2[focal]
    # Re-optimise G1 with b[focal] fixed to (b_G1[focal]+b_G2[focal])/2
    b_mean_focal = (b_G1[focal] + b_G2[focal]) / 2

    def nll_constrained(b_free, grp_X, focal_idx, b_fixed):
        b_full = b_free.copy()
        b_full[focal_idx] = b_fixed - b_free.mean()   # approx constraint
        return cml_nll(b_full, grp_X)

    # Simple approach: compute LRT as difference in CML per group with and without DIF param
    # Use the b-difference z-test as a proxy for LRT chi-square
    # LRT ~= z^2 for single df (Wald approximation)
    lrt_chi2[focal] = z_scores[focal]**2    # Wald approximation (z^2 ~ chi2(1))
    lrt_p[focal]    = 1 - chi2.cdf(lrt_chi2[focal], df=1)

print(f"\n{'Item':>5} {'LRT chi2':>10} {'p_LRT':>8} {'Δb':>7}  {'Sig':>5} {'True DIF':>9}")
print("-" * 55)
for i in range(I):
    sig = '***' if lrt_p[i] < 0.001 else ('**' if lrt_p[i] < 0.01
          else ('*' if lrt_p[i] < 0.05 else ''))
    true_dif = DIF_deltas[DIF_items.tolist().index(i)] if i in DIF_items else 0.0
    flag = " ◀" if i in DIF_items else ""
    print(f"{i+1:5d} {lrt_chi2[i]:10.3f} {lrt_p[i]:8.4f} "
          f"{delta_b[i]:7.3f}  {sig:>5}  {true_dif:+.2f}{flag}")

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
colors_lrt = ['#d62728' if p < 0.001 else '#ff7f0e' if p < 0.05 else 'steelblue'
              for p in lrt_p]
neg_log_p_lrt = -np.log10(np.maximum(lrt_p, 1e-10))
ax.bar(np.arange(1, I+1), neg_log_p_lrt, color=colors_lrt, alpha=0.75, edgecolor='white')
ax.axhline(-np.log10(0.05),  color='orange', linestyle='--', linewidth=1.2, label='p=0.05')
ax.axhline(-np.log10(0.001), color='red',    linestyle='--', linewidth=1.2, label='p=0.001')
ax.set_xlabel('Item'); ax.set_ylabel('−log₁₀(p_LRT)')
ax.set_title('LRT DIF (Wald approx.) — Significance per Item  (red=p<0.001)', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'dif_lrt.png'), dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ─── Section 7: Bayesian DIF Detection ─────────────────────────────────────

STAN_DIF = '''
data {
  int<lower=1> J1; int<lower=1> J2; int<lower=1> I;
  array[J1, I] int<lower=0,upper=1> X1;
  array[J2, I] int<lower=0,upper=1> X2;
}
parameters {
  vector[J1] theta1; vector[J2] theta2;
  vector[I]  b_raw1; vector[I]  b_raw2;
}
transformed parameters {
  vector[I] b1 = b_raw1 - mean(b_raw1);
  vector[I] b2 = b_raw2 - mean(b_raw2);
}
model {
  theta1 ~ normal(0,1); theta2 ~ normal(0,1);
  b_raw1 ~ normal(0,1); b_raw2 ~ normal(0,1);
  for (j in 1:J1) for (i in 1:I) X1[j,i] ~ bernoulli_logit(theta1[j]-b1[i]);
  for (j in 1:J2) for (i in 1:I) X2[j,i] ~ bernoulli_logit(theta2[j]-b2[i]);
}
generated quantities {
  vector[I] delta_b = b1 - b2;
}
'''

if STAN_AVAILABLE:
    stan_f = os.path.join(tmpdir, 'rasch_dif.stan')
    with open(stan_f, 'w') as f: f.write(STAN_DIF)
    model = cmdstanpy.CmdStanModel(stan_file=stan_f)
    fit = model.sample(
        data={'J1':J1,'J2':J2,'I':I,'X1':X1.tolist(),'X2':X2.tolist()},
        chains=4, iter_warmup=1000, iter_sampling=1000,
        show_progress=False, show_console=False)
    delta_post = fit.stan_variable('delta_b')      # (4000, I)
    b1_post    = fit.stan_variable('b1')
    b2_post    = fit.stan_variable('b2')
    print("Stan sampling complete.")
else:
    # Synthetic posterior: separate CML estimates + realistic uncertainty
    n_s = 2000
    b1_post = b_G1[None,:] + np.random.normal(0, se_G1[None,:], (n_s, I))
    b2_post = b_G2[None,:] + np.random.normal(0, se_G2[None,:], (n_s, I))
    # Add per-sample centering
    b1_post -= b1_post.mean(axis=1, keepdims=True)
    b2_post -= b2_post.mean(axis=1, keepdims=True)
    # Linking constant per sample
    link_s   = b1_post.mean(axis=1, keepdims=True) - b2_post.mean(axis=1, keepdims=True)
    delta_post = b1_post - (b2_post + link_s)
    print("Using synthetic posterior (Stan not available).")

delta_mean = delta_post.mean(axis=0)
delta_lo   = np.percentile(delta_post, 2.5,  axis=0)
delta_hi   = np.percentile(delta_post, 97.5, axis=0)
delta_sd   = delta_post.std(axis=0)

# Posterior probability of DIF: P(|delta_b| > 0.43)
pp_dif = (np.abs(delta_post) > 0.43).mean(axis=0)

print(f"\n{'Item':>5} {'Δb post.mean':>13} {'95% CI':>20} {'P(|Δ|>0.43)':>13} {'DIF?':>5}")
print("-" * 60)
for i in range(I):
    dif_flag = 'YES' if pp_dif[i] > 0.90 else ('??' if pp_dif[i] > 0.75 else '')
    true_dif = DIF_deltas[DIF_items.tolist().index(i)] if i in DIF_items else 0.0
    flag2 = " ◀" if i in DIF_items else ""
    print(f"{i+1:5d} {delta_mean[i]:13.3f} "
          f"[{delta_lo[i]:+.3f}, {delta_hi[i]:+.3f}]  "
          f"{pp_dif[i]:13.3f}  {dif_flag:>5}  {true_dif:+.2f}{flag2}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Bayesian DIF Detection — Posterior of Δb = b_G1 − b_G2', fontsize=13, fontweight='bold')

colors_b = ['#d62728' if pp_dif[i]>0.90 else '#ff7f0e' if pp_dif[i]>0.75 else 'steelblue'
            for i in range(I)]

ax = axes[0]
ax.bar(np.arange(1,I+1), delta_mean, color=colors_b, alpha=0.6, edgecolor='white')
ax.errorbar(np.arange(1,I+1), delta_mean,
            yerr=[delta_mean-delta_lo, delta_hi-delta_mean],
            fmt='none', color='black', capsize=4, linewidth=1.2)
ax.axhline(0,      color='black',  linestyle='--', linewidth=1.2)
ax.axhline( 0.43,  color='orange', linestyle=':', linewidth=1.0, label='±0.43 (ETS B)')
ax.axhline(-0.43,  color='orange', linestyle=':', linewidth=1.0)
ax.axhline( 0.64,  color='red',    linestyle=':', linewidth=1.0, label='±0.64 (ETS C)')
ax.axhline(-0.64,  color='red',    linestyle=':', linewidth=1.0)
ax.set_xlabel('Item'); ax.set_ylabel('Posterior mean Δb ± 95% CI')
ax.set_title('Posterior b-Difference per Item', fontsize=11); ax.legend(fontsize=9)

ax2 = axes[1]
ax2.bar(np.arange(1,I+1), pp_dif, color=colors_b, alpha=0.75, edgecolor='white')
ax2.axhline(0.90, color='red',    linestyle='--', linewidth=1.2, label='P=0.90 (strong DIF)')
ax2.axhline(0.75, color='orange', linestyle='--', linewidth=1.2, label='P=0.75 (moderate)')
ax2.set_xlabel('Item'); ax2.set_ylabel('P(|Δb| > 0.43)')
ax2.set_title('Posterior Probability of Meaningful DIF', fontsize=11); ax2.legend(fontsize=9)
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'dif_bayes.png'), dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ─── Section 8: DIF Detection Summary ──────────────────────────────────────

# Collect all method results
mh_ets  = ['A' if abs(d)<1.0 else ('B' if abs(d)<1.5 else 'C') for d in mh_delta]
bay_ets = ['C' if pp_dif[i]>0.90 else ('B' if pp_dif[i]>0.75 else 'A') for i in range(I)]

print("="*76)
print(f"{'Item':>5} | {'True Δb':>8} | {'b-diff ETS':>10} | {'MH ETS':>7} | "
      f"{'LRT p':>8} | {'Bayes ETS':>9} | {'Agreement':>9}")
print("="*76)
for i in range(I):
    true_dif = DIF_deltas[DIF_items.tolist().index(i)] if i in DIF_items else 0.0
    agree = "✓" if (ets_labels[i]!='A') == (i in DIF_items) else "✗"
    print(f"{i+1:5d} | {true_dif:+8.2f} | {ets_labels[i]:>10} | {mh_ets[i]:>7} | "
          f"{lrt_p[i]:8.4f} | {bay_ets[i]:>9} | {agree:>9}")

# Summary comparison plot
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('DIF Detection Summary — All Methods vs Ground Truth (1반 vs 2반)',
             fontsize=13, fontweight='bold')

truth_delta = np.zeros(I)
for idx, d in zip(DIF_items, DIF_deltas):
    truth_delta[idx] = d

methods = [
    (axes[0], truth_delta,   'Ground Truth Δb',      'black'),
    (axes[1], delta_b,       'b-Difference (CML)',   'steelblue'),
    (axes[2], mh_delta/2.35, 'Mantel-Haenszel (scaled)', 'darkorange'),
    (axes[3], delta_mean,    'Bayesian Posterior Δb', 'seagreen'),
]
for ax, vals, title, color in methods:
    cols = ['#d62728' if abs(v)>=0.64 else '#ff7f0e' if abs(v)>=0.43 else color
            for v in vals]
    ax.bar(np.arange(1,I+1), vals, color=cols, alpha=0.8, edgecolor='white')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axhline( 0.64, color='red',    linestyle=':', linewidth=0.8, alpha=0.6)
    ax.axhline(-0.64, color='red',    linestyle=':', linewidth=0.8, alpha=0.6)
    ax.axhline( 0.43, color='orange', linestyle=':', linewidth=0.8, alpha=0.6)
    ax.axhline(-0.43, color='orange', linestyle=':', linewidth=0.8, alpha=0.6)
    ax.set_title(title, fontsize=11); ax.set_xlabel('Item')

plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'dif_summary.png'), dpi=120, bbox_inches='tight')
plt.show()


---
<a id="sec8"></a>
## 8. DIF Interpretation and Practical Recommendations

### Which Items to Flag

An item warrants review if **two or more methods** agree on DIF:

| Decision Rule | Action |
|---|---|
| All 4 methods = C | Remove item immediately; do not use in scoring |
| 3 of 4 methods = B/C | Flag for expert review; may be biased |
| 2 of 4 methods = B/C | Conditional flag; review item content for plausible bias source |
| Only 1 method = B/C | Monitor; likely statistical noise, especially with N<50 per group |
| All = A | Accept item as functioning equivalently across groups |

### Post-DIF Procedures in RMM

1. **Identify DIF items** using the above rules
2. **Expert content review**: does the item contain construct-irrelevant content that would advantage one class?
3. **Distinguish bias from impact**:
   - Bias: the item is unfair (e.g., assumes knowledge taught only in one class) → remove from scoring
   - Impact: the item reflects a real trait difference → retain but report separately
4. **Re-anchor**: remove DIF items from the linking anchor set; re-estimate item difficulties
5. **Report**: flag DIF items in test documentation; adjust scoring if necessary

### Cautions for Small Classes (N ≤ 40 per group)

- **Power is low**: large DIF ($|\Delta b| > 0.8$) may be missed; moderate DIF ($|\Delta b| \approx 0.4$) will rarely be detected
- **Type I error inflation**: random variation in N=40 can produce apparent DIF of ±0.4 logits
- **Recommendation**: Use the Bayesian method (posterior credible interval) for small groups — it naturally incorporates uncertainty and avoids sharp cutoff problems

### (한국어) DIF 해석과 실무 권고사항

**DIF 탐지 결론**:
두 가지 이상의 방법이 일치하는 문항을 우선 검토 대상으로 삼습니다.

**1반-2반 DIF의 가장 흔한 원인 (우선순위)**:
1. 담당 교사가 달라 특정 유형의 문제를 다른 깊이로 다룬 경우 (A1, A2)
2. 수업 중 유사한 문제를 연습한 경우 (A5)
3. 시험 시행 조건 차이 — 시간, 프록터, 환경 (B1–B3)
4. 소표본(N≈30–40)에서의 무선 변동 (F1)

**작은 학급에서의 주의사항**:
- 소표본에서는 Bayesian 방법이 가장 안정적
- "한 방법에서만" DIF가 탐지된 경우는 통계적 잡음일 가능성 높음
- DIF 탐지 결과를 교사 간 수업 개선 논의에 활용
